# Time series, baselines and exponential smoothing

A **time series** is a sequence of values measured at successive points in time. Forecasting asks what the next values will be and how certain that is. This notebook covers the first block of Session 12. The first three sections cover what to do before any model: describe the series (trend, seasonality, autocorrelation), build it correctly from event or count data (aggregating to months with pandas), and compute the **baselines** that every later model must beat. The last three sections fit **exponential smoothing**, give its **prediction intervals** and end with a short outlook on **ARIMA**. The example is demand for short-term rentals in Berlin, measured by the number of guest reviews written per month on Airbnb, from the case-study table `case-study/data/airbnb/reviews_monthly.parquet` (Inside Airbnb, snapshot of 26 June 2026; prepare it with `uv run python case-study/prepare_airbnb.py`).

Reviews are a **proxy** for stays: not every guest writes a review, and the share who do can change over time. A city administration, a tourism board or a host who plans prices and cleaning staff would like to know the number of stays; the number of reviews is the closest public measure. The series has a clear trend, a strong summer season and a deep break during the COVID-19 pandemic, which makes it a good teaching series.

The code cells build on each other; run them in order. The first cell moves to the repository root, where the case-study paths start.

```mermaid
flowchart LR
    E["Count table<br/>(listing x month)"] --> A["Aggregate to one<br/>regular series"]
    A --> P["Plot and describe:<br/>trend, seasonality, ACF"]
    P --> B["Baselines:<br/>naive, seasonal naive,<br/>moving average"]
    B --> M["Exponential smoothing,<br/>ARIMA, lag features"]
    M --> V["Backtest against<br/>the baselines"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Time series: trend, seasonality and autocorrelation

### Concept

We write the series as y₁, y₂, …, y_T, where t counts the periods (here months). Most series combine three **components**:

- **Trend**: the long-run level and its direction (growth, decline, a change of direction).
- **Seasonality**: a pattern that repeats with a fixed **period** m, for example m = 12 for monthly data with a yearly cycle, m = 7 for daily data with a weekly cycle.
- **Remainder**: what is left, the irregular part.

In an **additive** decomposition y = trend + seasonal + remainder. In a **multiplicative** one y = trend × seasonal × remainder, which fits when the seasonal swings grow with the level. Taking logarithms turns a multiplicative series into an additive one. **STL** (seasonal–trend decomposition using LOESS, a local regression) estimates the components; with `robust=True` it gives little weight to extreme months, so that a one-off shock does not distort the seasonal pattern.

**Autocorrelation** at lag k is the correlation between the series and itself shifted by k periods, corr(y_t, y_{t−k}). The **autocorrelation function** (ACF) lists it for k = 1, 2, …. Small worked example: the series 2, 4, 6, 8 has a lag-1 pairing (2, 4), (4, 6), (6, 8), a perfect linear relation: any trend makes the low-lag autocorrelations large. To see seasonality, remove the trend first, for example by **differencing** (y_t − y_{t−1}); for log counts the difference is roughly the monthly growth rate.

The series used in this session sums the reviews of all Berlin listings per month from January 2016 to May 2026 (125 months). Three features are visible at once: growth from about 1,000 reviews a month in early 2016 to more than 13,000 in summer 2025, a summer peak every year, and a collapse in spring 2020: 4,865 reviews in July 2019, 285 in April 2020, when tourist stays were banned, and a slow recovery until late 2021.

![Four stacked panels for 2016-2026: the log monthly review counts with a deep dip in 2020-2021, a rising trend that falls during the pandemic, a stable seasonal pattern with a summer peak, and a remainder with large negative values in spring 2020 and early 2021](../figures/decomposition.png)

![Two ACF plots: for the log counts of the whole period the bars are high and decay slowly (trend); for the monthly growth rates since 2022 lag 12 stands out at 0.64](../figures/acf.png)

### Why it matters

The components say what a forecasting model must capture. A model that ignores a summer peak of +24 % or a change of level is wrong by construction. Autocorrelation is the reason time series need their own methods: neighbouring months are not independent observations, so random train–test splits and ordinary cross-validation do not apply (lecture of block 2). A break such as the pandemic raises a further question that this session returns to several times: which part of the history still describes the future?

### How it works in Python

In [2]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf

reviews = pd.read_parquet("case-study/data/airbnb/reviews_monthly.parquet")   # listing_id, month, n_reviews
y = reviews.groupby("month")["n_reviews"].sum()["2016":"2026-05"].rename("reviews")
y.index.freq = "MS"                                   # monthly, dated at the month start
log_y = np.log(y)
print(len(y), y.loc[["2019-07-01", "2020-04-01", "2025-07-01"]].to_list())   # 125 [4865, 285, 13132]

stl = STL(log_y, period=12, seasonal=13, robust=True).fit()   # trend + seasonal + remainder = log_y
recent = stl.seasonal["2022":"2025"]
factor = np.exp(recent).groupby(recent.index.month).mean()     # multiplicative seasonal factors
print(factor.round(2).to_dict())
# {1: 0.71, 2: 0.74, 3: 0.89, 4: 1.03, 5: 1.18, 6: 1.24, 7: 1.24, 8: 1.14, 9: 1.23, 10: 1.14, 11: 0.88, 12: 0.81}
print(round(np.exp(stl.resid["2020-04-01"]), 2))               # 0.1: April 2020 at 10 % of the expected value

print(acf(log_y, nlags=12)[[1, 6, 12]].round(2))                      # [0.91 0.57 0.49]  trend dominates
print(acf(log_y["2022":].diff().dropna(), nlags=12)[[1, 6, 12]].round(2))   # [ 0.41 -0.3   0.64]  yearly pattern
# plots: stl.plot(); statsmodels.graphics.tsaplots.plot_acf(log_y["2022":].diff().dropna(), lags=24)

125 [4865, 285, 13132]
{1: 0.71, 2: 0.74, 3: 0.89, 4: 1.03, 5: 1.18, 6: 1.24, 7: 1.24, 8: 1.14, 9: 1.23, 10: 1.14, 11: 0.88, 12: 0.81}
0.1
[0.91 0.57 0.49]
[ 0.41 -0.3   0.64]


Since 2022, June, July and September have about 24 % more reviews than the trend; January and February about 27 % fewer. Berlin's tourist season explains this reading, but the data alone cannot show the cause. The STL remainder puts April 2020 at a tenth of the expected value: the pandemic is a shock that no seasonal pattern describes. On the log counts of the whole period, the ACF is high at every lag because of the trend. After differencing the post-pandemic months, lag 12 stands out with 0.64: the yearly pattern is strong compared with the month-to-month noise.

### In practice

- **Official statistics.** Eurostat and the national statistical offices publish seasonally adjusted figures, computed with X-13ARIMA-SEATS or TRAMO/SEATS (for example in JDemetra+). The Statistical Office for Berlin-Brandenburg publishes monthly guest and overnight-stay figures for accommodation establishments.
- **Tourism and hospitality.** Hotels and short-term rental hosts set prices and plan cleaning and staff around the seasonal peak; revenue-management systems start from a decomposition like the one above.
- **Retail.** Staff and stock are planned around the seasonal peak of the year-end holidays.

> **Warning.** **A trend makes the ACF look strong at every lag.** High autocorrelations of a trending series are not evidence of seasonality or of a useful predictor. Difference first.

> **Caution.** **A decomposition describes, it does not explain.** The January factor of 0.71 says *that* January is low, not *why*. And a robust decomposition only downweights the pandemic months; it does not tell you whether the world after the break is the same as before.

> 📖 **Read more:** [extras/01-stl-decomposition.ipynb](extras/01-stl-decomposition.ipynb) (statsmodels examples): STL on the CO₂ series in detail: the lengths of the seasonal and trend smoothers, robust fitting against outliers, the LOESS degree, and forecasting with `STLForecast`.

## Aggregating to regular time intervals with pandas

### Concept

Forecasting methods expect one value per period at a fixed **frequency**. Event data (one row per review, order or call) become a series by counting or summing per period. In pandas the timestamps go into a `DatetimeIndex`, and **`resample`** groups them into calendar periods: `"D"` days, `"W"` weeks ending on Sunday, `"MS"` months labelled by their first day, `"QS"` quarters, `"YS"` years. An aggregation (`size`, `sum`, `mean`) then gives one value per period.

The case-study table `reviews_monthly` is already aggregated one step: one row per listing and month with at least one review. Summing over listings gives the city-wide series; selecting a listing or a district gives one of thousands of smaller series. The table `calendar` has one row per listing and day for the next 365 days (is the night still bookable?), a daily series that can be resampled to weeks or months.

Small example: 3 and 5 reviews on 30 and 31 January, 2 and 4 on 1 and 3 February give 8 for January and 6 for February.

### Why it matters

The frequency is a modelling decision. It sets what can be forecast (a monthly staff plan needs monthly values), how noisy each value is (daily counts are noisier) and which seasonal patterns are visible. Many forecasting errors start with a badly built series: missing periods, a partial first or last period, a series whose definition changes over time.

### How it works in Python

In [3]:
tiny = pd.Series([3, 5, 2, 4], index=pd.to_datetime(["2026-01-30", "2026-01-31", "2026-02-01", "2026-02-03"]))
print(tiny.resample("MS").sum().to_dict())
# {Timestamp('2026-01-01 00:00:00'): 8, Timestamp('2026-02-01 00:00:00'): 6}

# the last months: the snapshot was scraped between 26 June and 3 July 2026
full = reviews.groupby("month")["n_reviews"].sum()
print(full.loc["2026-04":].to_dict())
# {Timestamp('2026-04-01 00:00:00'): 11720, Timestamp('2026-05-01 00:00:00'): 15024,
#  Timestamp('2026-06-01 00:00:00'): 9968, Timestamp('2026-07-01 00:00:00'): 22}
print(full.groupby(full.index.year).sum().loc[[2016, 2019, 2020, 2021, 2025]].to_dict())
# {2016: 17829, 2019: 53965, 2020: 26499, 2021: 35447, 2025: 130357}

# one listing: months without reviews are missing rows, not zeros
one = reviews[reviews["listing_id"] == reviews["listing_id"].iloc[0]].set_index("month")["n_reviews"]["2016":"2026-05"]
dense = one.reindex(pd.date_range("2016-01-01", "2026-05-01", freq="MS"), fill_value=0)
print(len(one), len(dense), round(one.mean(), 2), round(dense.mean(), 2))   # 20 125 1.6 0.26

# a daily series: share of listings still bookable on each future night, resampled to months
calendar = pd.read_parquet("case-study/data/airbnb/calendar.parquet")      # listing_id, date, available
open_share = calendar.groupby("date")["available"].mean()
print(open_share.resample("MS").mean().round(2).iloc[[0, 1, 6, 12]].to_dict())
# {Timestamp('2026-06-01 00:00:00'): 0.13, Timestamp('2026-07-01 00:00:00'): 0.31,
#  Timestamp('2026-12-01 00:00:00'): 0.48, Timestamp('2027-06-01 00:00:00'): 0.36}

{Timestamp('2026-01-01 00:00:00'): 8, Timestamp('2026-02-01 00:00:00'): 6}
{Timestamp('2026-04-01 00:00:00'): 11720, Timestamp('2026-05-01 00:00:00'): 15024, Timestamp('2026-06-01 00:00:00'): 9968, Timestamp('2026-07-01 00:00:00'): 22}
{2016: 17829, 2019: 53965, 2020: 26499, 2021: 35447, 2025: 130357}
20 125 1.6 0.26
{Timestamp('2026-06-01 00:00:00'): 0.13, Timestamp('2026-07-01 00:00:00'): 0.31, Timestamp('2026-12-01 00:00:00'): 0.48, Timestamp('2027-06-01 00:00:00'): 0.36}


Four lessons. First, the snapshot was scraped at the end of June 2026, so June has only part of its reviews (9,968) and July almost none (22): the series must end with the last complete month, May 2026. Second, a review table only knows the listings that still exist on the scraping date. Listings that left Airbnb before June 2026 took their reviews with them, so the early years are undercounted, by an unknown amount, and the growth since 2016 is partly an artefact of this **survivorship**. Third, the first listing of the table has reviews in only 20 of 125 months: a mean computed without the empty months (1.6) is six times the true monthly mean (0.26). `reindex` with `fill_value=0` (or `resample(...).sum()` on event data) keeps the empty months. Fourth, the calendar shows that only 13 % of the nights in the last days of June 2026 were still bookable, against 48 % for December 2026. This is not a forecast of demand: far-away nights are open because they have not been booked or blocked *yet*. The first and last calendar months are also partial (26–30 June 2026, 1–2 July 2027).

### In practice

- **Electricity grid operators** forecast load per 15 minutes or per hour for balancing, and per month for planning; the frequency follows the decision.
- **Public health agencies** count notified cases per week, as in the weekly reports of the Robert Koch Institute, partly because daily counts are distorted by reporting delays at weekends.
- **Retailers** forecast sales per product and store per day; the M5 competition used Walmart unit sales at this level.

> **Warning.** **The last period may not be complete.** A month that is not over yet, or a dataset that was scraped in the middle of a month, has a low last value that is not a real drop. Cut incomplete periods before modelling.

> **Tip.** Use `resample(...).size()` for counts of events and `.sum()` for amounts or pre-aggregated counts. For time zones, convert timestamps with `tz_convert` before resampling to days.

## Baselines: naive, seasonal naive and moving average

### Concept

A **baseline** is a forecast so simple that any proposed model must beat it to be worth using. The three standard ones, for a forecast **horizon** of h periods after the last observed value y_T:

- **Naive**: every future value equals the last observation, ŷ_{T+h} = y_T. It is the best possible forecast for a random walk (for example many share prices).
- **Seasonal naive**: every future value equals the value one season earlier, ŷ_{T+h} = y_{T+h−m}. With monthly data, next July equals last July.
- **Moving average**: the mean of the last n observations, ŷ_{T+h} = (y_T + … + y_{T−n+1}) / n. It smooths noise but lags behind changes.

For a growing series, a common business variant is the **seasonal naive forecast times growth**: last year's value of the same month multiplied by the growth of the last twelve months over the twelve months before. It is still a rule anyone can compute in a spreadsheet.

Errors are measured on a **hold-out period**: the last part of the series, never used to fit. Here the hold-out period is the last twelve complete months, June 2025 to May 2026. The **mean absolute error** MAE = mean |y_t − ŷ_t| is in the units of the series ("on average 1,000 reviews off per month"). Scaled errors that compare across series follow in the lecture of block 2.

Worked example: the last three training months, March to May 2025, had 9,135, 10,575 and 11,977 reviews. Naive forecasts 11,977 for every month; the 3-month moving average forecasts (9,135 + 10,575 + 11,977) / 3 ≈ 10,562; seasonal naive forecasts June 2025 = June 2024 = 11,837 and December 2025 = December 2024 = 7,899. The twelve months up to May 2025 had 26 % more reviews than the twelve months before, so the growth variant forecasts June 2025 as 11,837 × 1.26 ≈ 14,900.

![Line chart of monthly reviews 2022-2026 with four dashed baseline forecasts for June 2025 to May 2026: naive and moving average as flat lines, seasonal naive repeating the previous year's pattern below the actual values, and seasonal naive times growth closest to the actual values but too high in summer 2025](../figures/baselines.png)

### Why it matters

Without a baseline, a complex model can look useful while being worse than "same as last year". Baselines also show what kind of series one has: when the naive forecast is hard to beat, the series behaves like a random walk and there is little to learn from its own past. Here every baseline has an obvious flaw: the flat ones ignore the season, plain seasonal naive ignores the growth.

### How it works in Python

In [4]:
train, test = y[:"2025-05"], y["2025-06":]           # hold out the last 12 complete months
h = len(test)

naive = pd.Series(train.iloc[-1], index=test.index)                     # last value
snaive = pd.Series(train.iloc[-12:].to_numpy(), index=test.index)       # same month last year
moving = pd.Series(train.iloc[-3:].mean(), index=test.index)            # mean of the last 3 months
growth = train.iloc[-12:].sum() / train.iloc[-24:-12].sum()             # last 12 months vs the 12 before
snaive_growth = snaive * growth
print(round(growth, 2))                                                 # 1.26


def mae(actual, forecast):
    return np.mean(np.abs(np.asarray(actual) - np.asarray(forecast)))


for name, fc in {"naive": naive, "seasonal naive": snaive, "moving average": moving,
                 "seasonal naive x growth": snaive_growth}.items():
    print(f"{name:24s} MAE {mae(test, fc):5.0f}")
# naive                    MAE  1710
# seasonal naive           MAE  1586
# moving average           MAE  1935
# seasonal naive x growth  MAE  1038

1.26
naive                    MAE  1710
seasonal naive           MAE  1586
moving average           MAE  1935
seasonal naive x growth  MAE  1038


In the hold-out year the seasonal naive forecast beats the flat forecasts, but it is still about 1,600 reviews per month too low (14 % of the mean of 11,466), because the market grew. Multiplying by last year's growth cuts the error to about 1,000. One hold-out year is a single test, though: the lecture of block 2 repeats the comparison from six forecast origins.

### In practice

- **Forecasting competitions.** In the M4 competition (100,000 series, 2018) many submitted methods did not beat simple statistical benchmarks; the naive and seasonal naive forecasts are the reference points of the error measures.
- **Demand planning.** Planners report *forecast value added*: the improvement of the official forecast over a naive forecast. Negative values are common, for example when manual adjustments make forecasts worse.
- **Finance.** For exchange rates and share prices, the random-walk (naive) forecast is notoriously hard to beat, as Meese and Rogoff (1983) showed for exchange-rate models.

> **Warning.** **Never evaluate on random months.** A random split puts later months into training and lets the model use the future to "predict" the past.

> **Caution.** **Avoid MAPE for small counts.** The mean absolute percentage error divides by the actual value and explodes when values are close to zero, as for a single listing or for April 2020 (285 reviews). Use MAE or the scaled MASE (lecture of block 2).

> **Note.** **Ethics.** Inside Airbnb scrapes public listing pages. Work with aggregates such as the city-wide series, and do not single out or name hosts.

*Worked example:* part 1 of [02-case-study-airbnb-review-forecast.ipynb](02-case-study-airbnb-review-forecast.ipynb) builds the monthly number of reviews in Berlin, checks its quality and computes the baseline forecasts.

## From baselines to exponential smoothing

The baselines above use either the last value or a plain average. **Exponential smoothing** sits in between: it averages all past values with weights that decrease the older a value is, and it extends naturally to trend and seasonality. Its statistical form, the **ETS** model, also gives **prediction intervals**: a range that should contain the future value with a stated probability. The last section is a short outlook on **ARIMA**, the other classical family of forecasting models, and shows one of its practical advantages: it can skip a period of missing values, such as the pandemic months. The next code cell reloads the monthly series (January 2016 to May 2026, hold-out year June 2025 to May 2026), so this part can also be run on its own after the first cell.

```mermaid
flowchart TD
    S["Simple exponential smoothing<br/>(level only)"] --> H["Holt's method<br/>(+ trend)"]
    H --> D["Damped trend<br/>(trend flattens out)"]
    H --> HW["Holt-Winters<br/>(+ seasonality)"]
    D --> ETS["ETS models:<br/>Error, Trend, Seasonal<br/>with prediction intervals"]
    HW --> ETS
```

In [5]:
import numpy as np
import pandas as pd
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

reviews = pd.read_parquet("case-study/data/airbnb/reviews_monthly.parquet")
y = reviews.groupby("month")["n_reviews"].sum()["2016":"2026-05"].rename("reviews")
y.index.freq = "MS"
train, test = y[:"2025-05"], y["2025-06":]
h = len(test)


def mae(actual, forecast):
    return np.mean(np.abs(np.asarray(actual) - np.asarray(forecast)))

## Exponential smoothing

### Concept

**Simple exponential smoothing** (SES) keeps one number, the **level** ℓ, and updates it after each observation:

ℓ_t = α·y_t + (1 − α)·ℓ_{t−1},  forecast ŷ_{T+h} = ℓ_T for every horizon h.

The **smoothing parameter** α (between 0 and 1) sets how fast the level reacts. Unrolling the formula shows that the weights on past values are α, α(1 − α), α(1 − α)², …: they decay exponentially. α = 1 gives the naive forecast; a small α gives a long, smooth average.

Worked example with α = 0.5, the series 10, 12, 11, 15 and a starting level of 10:

| t | y_t | ℓ_t = 0.5·y_t + 0.5·ℓ_{t−1} |
|---|---|---|
| 1 | 10 | 0.5·10 + 0.5·10 = 10 |
| 2 | 12 | 0.5·12 + 0.5·10 = 11 |
| 3 | 11 | 0.5·11 + 0.5·11 = 11 |
| 4 | 15 | 0.5·15 + 0.5·11 = 13 |

The forecast for t = 5, 6, … is 13.

Extensions add further smoothed components, each with its own parameter:

- **Holt's method** adds a **trend** b_t (the current slope); the forecast is ℓ_T + h·b_T, a straight line.
- A **damped trend** multiplies the slope by φ < 1 for each further step, so long-horizon forecasts flatten out instead of growing forever. Damped trends are a robust default for business series.
- **Holt–Winters** adds a **seasonal** component s_t with period m, added (additive) or multiplied (multiplicative).

The parameters are estimated by minimising the one-step-ahead forecast errors on the training data.

### Why it matters

Exponential smoothing is fast, needs little data, and its components (level, trend, seasonal pattern) can be explained to non-specialists. It performs well in forecasting competitions: in the M3 and M4 competitions, exponential smoothing methods and their combinations were among the strongest statistical approaches. Its weakness shows on a series with a break: the model has no notion of "this period was exceptional"; it adapts its level to whatever happened.

### How it works in Python

In [6]:
# the worked example
tiny = pd.Series([10.0, 12.0, 11.0, 15.0])
ses = SimpleExpSmoothing(tiny, initialization_method="known", initial_level=10).fit(
    smoothing_level=0.5, optimized=False)
print(ses.forecast(2).to_list())          # [13.0, 13.0]

# damped trend + additive seasonality on log counts (multiplicative on the original scale),
# fitted once on the whole history and once on the months since January 2022
for start in ("2016", "2022"):
    ets = ETSModel(np.log(train[start:]), error="add", trend="add", damped_trend=True,
                   seasonal="add", seasonal_periods=12).fit(disp=False)
    params = dict(zip(ets.param_names[:4], ets.params[:4].round(3)))
    print(start, params, f"MAE {mae(test, np.exp(ets.forecast(h))):.0f}")
# 2016 {'smoothing_level': 1.0, 'smoothing_trend': 0.0, 'smoothing_seasonal': 0.0, 'damping_trend': 0.98} MAE 1952
# 2022 {'smoothing_level': 0.175, 'smoothing_trend': 0.175, 'smoothing_seasonal': 0.0, 'damping_trend': 0.869} MAE 851

[13.0, 13.0]
2016 {'smoothing_level': 1.0, 'smoothing_trend': 0.0, 'smoothing_seasonal': 0.0, 'damping_trend': 0.98} MAE 1952


2022 {'smoothing_level': 0.175, 'smoothing_trend': 0.175, 'smoothing_seasonal': 0.0, 'damping_trend': 0.869} MAE 851


The choice of history decides the result. On the whole history the model needs α = 1 to follow the collapse and recovery of 2020–2021: it becomes a seasonal version of the naive forecast and is worse than every baseline above (MAE 1,952). Fitted on the months since January 2022, after the recovery, α = 0.18 means the level moves moderately, the trend is damped (φ = 0.87) and the seasonal pattern is fixed (γ = 0). Its MAE of 851 beats the best baseline of the baselines section, seasonal naive times growth (1,038). Starting a model after a break is a common, defensible choice; its cost is a short history (41 months), which makes the estimates less certain. The lecture of block 2 checks whether the result holds from other forecast origins.

### In practice

- **Inventory and supply chain.** Planning systems forecast demand for thousands of products with exponential smoothing because it is cheap to fit and update every week; Gardner (2006) reviews decades of such use.
- **Forecasting competitions.** The M3 competition (3,003 series) and the M4 competition (100,000 series) found that exponential smoothing and simple combinations were hard to beat (Makridakis and Hibon, 2000; Makridakis, Spiliotis and Assimakopoulos, 2020).
- **Open-source tools.** Nixtla's StatsForecast fits `AutoETS` models to millions of series in parallel; [extras notebook 03](extras/03-statsforecast-quickstart.ipynb) shows the library on the classic air passengers data (with `AutoARIMA`).

> **Warning.** **A non-damped trend extrapolates forever.** Holt's linear trend fitted on a growth phase forecasts unlimited growth. Prefer a damped trend unless there is a reason to expect the trend to continue.

> **Tip.** When seasonal swings grow with the level, fit an additive model on the **log** of the series and transform back with `np.exp`, as above. This keeps forecasts positive, too.

## Prediction intervals

### Concept

A **point forecast** is a single number. A **prediction interval** gives a range that should contain the future value with a stated probability, for example 95 %. For the simplest ETS model (SES with normal errors of standard deviation σ), the 95 % interval h steps ahead is

ŷ_{T+h} ± 1.96 · σ · √(1 + (h − 1)·α²).

The interval widens with h, because the uncertainty about the level accumulates. Example: with σ = 500 reviews and α = 0.2, the half-width is 1.96·500 = 980 for h = 1 and 1.96·500·√(1 + 11·0.04) = 980·1.2 ≈ 1,176 for h = 12.

A prediction interval is not a **confidence interval**. A confidence interval describes uncertainty about a parameter (for example the mean number of reviews per month); a prediction interval describes where a single future observation will fall, so it is much wider.

A 95 % interval is **calibrated** if about 95 % of future values fall inside it. This **coverage** can be checked in a backtest (lecture of block 2).

![Line chart of monthly reviews 2022 to 2026 with the ETS point forecast for June 2025 to May 2026 and a shaded 95 % interval that follows the seasonal pattern and widens towards May 2026; November and December 2025 lie above the interval](../figures/ets_interval.png)

### Why it matters

Decisions depend on the range more than on the point. A host or a cleaning company that plans staff for the summer needs a high but plausible number of turnovers, not the average. A city office that monitors short-term rentals wants to know whether a change is within normal variation. A forecast without an interval hides the most important information: how wrong it may be.

### How it works in Python

In [7]:
pred = ets.get_prediction(start=test.index[0], end=test.index[-1])   # the model fitted since 2022
frame = np.exp(pred.summary_frame(alpha=0.05))      # columns mean, pi_lower, pi_upper (back on counts)
print(frame.iloc[[0, 6, 11]][["mean", "pi_lower", "pi_upper"]].round(0))
#                mean  pi_lower  pi_upper
# 2025-06-01  12598.0   11617.0   13662.0
# 2025-12-01   8253.0    7112.0    9578.0
# 2026-05-01  12473.0    9861.0   15776.0
inside = (test >= frame["pi_lower"]) & (test <= frame["pi_upper"])
print(inside.sum(), "of", h, "months inside the 95 % interval")   # 10 of 12 months inside the 95 % interval
print(test[~inside].to_list())                                    # [10524, 9580]: November and December 2025

               mean  pi_lower  pi_upper
2025-06-01  12598.0   11617.0   13662.0
2025-12-01   8253.0    7112.0    9578.0
2026-05-01  12473.0    9861.0   15776.0
10 of 12 months inside the 95 % interval
[10524, 9580]


Ten of twelve months fall inside. November and December 2025 were higher than the model thought plausible: the winter of 2025/26 was stronger than the three winters the model had seen since 2022. The interval widens from about ±8 % in the first month to about ±25 % after a year. Intervals computed on the log scale are asymmetric on the original scale (the upper part is longer), which suits counts.

### In practice

- **Central banks.** The Bank of England publishes its inflation forecast as a "fan chart", in which shaded bands show the probability ranges; the format has been used since the 1990s.
- **Weather services.** Ensemble forecasts give probabilities and ranges ("70 % chance of rain"), and forecasters check their calibration systematically.
- **Capacity planning.** Call centres and hospitals plan staff for an upper quantile of the forecast demand, such as the 80th or 90th percentile.

> **Warning.** **Model-based intervals are usually too narrow.** They assume the model is correct and the future behaves like the past. Structural breaks (a pandemic, a new law) are not in the interval. In Berlin, registration numbers for short-term rentals have been required for years, and the EU regulation on short-term rental data (Regulation (EU) 2024/1028) applies since 20 May 2026; if enforcement removes unregistered listings, the series could drop in a way no interval of this notebook anticipates. Check coverage in a backtest, and say what the interval does not include.

> **Tip.** Conformal prediction gives intervals with empirical coverage from backtest errors, for any model including gradient boosting. StatsForecast and MLForecast implement it (`ConformalIntervals`), and the Nixtla tutorials show how.

## ARIMA as an outlook

### Concept

**ARIMA(p, d, q)** models describe the autocorrelation of a series directly:

- **I** (integrated, d): difference the series d times to remove the trend (d = 1: monthly changes);
- **AR** (autoregressive, p): the differenced value depends linearly on its own p previous values;
- **MA** (moving average, q): and on the q previous forecast errors (not the same as the moving-average baseline).

**Seasonal ARIMA** adds the same terms at the seasonal lag, written (P, D, Q)_m. The classic "airline model" (0, 1, 1)(0, 1, 1)₁₂, introduced by Box and Jenkins for airline passenger numbers, differences once at lag 1 and once at lag 12 and has one MA term of each kind. Choosing p, d and q is traditionally done with the ACF and partial ACF; automatic methods (`AutoARIMA`) compare candidate models with an information criterion (AIC).

ARIMA models in statsmodels are estimated as **state-space models** with the Kalman filter, which can step over **missing values**: a month marked as missing (NaN) is predicted, but not used to update the model. That offers a third way to handle the pandemic, besides keeping it (and letting it distort the model) or cutting the history: mark March 2020 to December 2021 as missing and keep the years before and after.

```mermaid
flowchart LR
    Y["Series"] --> D["Difference d times<br/>(and D times at lag m)"]
    D --> AR["AR: past values<br/>(p, P)"]
    D --> MA["MA: past errors<br/>(q, Q)"]
    AR --> F["Forecast and interval,<br/>undo differencing"]
    MA --> F
```

### Why it matters

ARIMA and exponential smoothing are the two standard statistical families for single series; each is equivalent to the other in some special cases, and neither dominates. ARIMA can include external predictors (ARIMAX, dynamic regression; FPP Chapter 10), for example a dummy variable for lockdown months. This course stops at an outlook; the models are covered in depth in FPP Chapter 9.

### How it works in Python

In [8]:
log_gap = np.log(train)
log_gap["2020-03":"2021-12"] = np.nan                     # treat the pandemic months as missing
airline = SARIMAX(log_gap, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
fc = airline.get_forecast(h)
arima_fc = np.exp(fc.predicted_mean)
print(f"airline model MAE {mae(test, arima_fc):.0f}")      # airline model MAE 450
interval = np.exp(fc.conf_int(alpha=0.05))                 # 95 % prediction interval
print(((test >= interval.iloc[:, 0]) & (test <= interval.iloc[:, 1])).sum(), "of 12 inside")   # 12 of 12 inside
print(((interval.iloc[:, 1] - interval.iloc[:, 0]) / arima_fc).round(2).iloc[[0, 11]].to_list())   # [0.27, 0.77]

full = SARIMAX(np.log(train), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
print(f"without the gap MAE {mae(test, np.exp(full.get_forecast(h).predicted_mean)):.0f}")   # without the gap MAE 2122

airline model MAE 450
12 of 12 inside
[0.27, 0.77]


without the gap MAE 2122


With the pandemic marked as missing, the airline model has the lowest hold-out error so far (MAE 450) and covers all twelve months; its interval is wider than that of ETS (27 % of the forecast in the first month, 77 % after a year), because it learns from the more volatile years before 2020 too. The same model on the series *with* the pandemic months is the worst of all (MAE 2,122): how a break is handled matters more than the choice between ETS and ARIMA. One year is one test, though; the lecture of block 2 backtests both. [Extras notebook 04](extras/04-arima.ipynb) (statsmodels) and the cross-validation of [extras notebook 06](extras/06-statsforecast-cross-validation.ipynb) (StatsForecast, with `AutoETS`) go further.

### In practice

- **Seasonal adjustment.** X-13ARIMA-SEATS, used by the US Census Bureau and many statistical offices, fits seasonal ARIMA models to extend series before adjusting them; for the pandemic, Eurostat's guidance of 2020 recommended treating the lockdown months as outliers, so that they would not change the seasonal factors.
- **Macroeconomic forecasting.** ARIMA models are a standard benchmark in central bank and research forecasting exercises.
- **Large-scale forecasting.** Nixtla reports that its `AutoARIMA` was more accurate and much faster than Prophet on large collections of series (StatsForecast experiments repository).

> **Caution.** **Differencing at lag 12 and lag 1 amplifies recent changes.** Seasonal ARIMA models can react strongly to the last year, which is good after a lasting change and bad after a one-off spike.

> **Warning.** **Marking months as missing is a modelling assumption.** It says that the pandemic months tell us nothing about the level, trend and season afterwards. That is plausible for a ban on tourist stays, but it should be stated in the report, together with the alternative (start after the break).

*Worked example:* part 2 of [02-case-study-airbnb-review-forecast.ipynb](02-case-study-airbnb-review-forecast.ipynb) fits exponential smoothing, decides how to treat the pandemic months and compares the models with the baselines.

## Check your understanding

1. Monthly reviews have a seasonal swing of ±1,000 at a level of 4,000 and ±3,000 at a level of 12,000. Additive or multiplicative decomposition? What transformation helps?
2. The ACF of the log counts is 0.91 at lag 1 and 0.49 at lag 12. Does this show a yearly pattern? What should you compute instead?
3. The snapshot was scraped at the end of June 2026. What happens to the series and to a naive forecast if June 2026 is kept?
4. Why can the growth of the review series since 2016 be overstated? Name the mechanism and one way to check its size.
5. Why must the hold-out period be at the end of the series?
6. Continue the worked SES example with y₅ = 9. What is ℓ₅ and the forecast for t = 6? What would it be with α = 0.9?
7. ETS fitted on the whole history estimates α = 1. What does this mean for its forecasts, and why does the pandemic push α there?
8. A 95 % interval contains 10 of 12 actual values in one year. Is it calibrated? What would you need to know more?
9. Explain to a host the difference between a confidence interval for the mean number of reviews per month and a prediction interval for next August.
10. Name three ways to handle the pandemic months in a forecasting model, and one assumption behind each.

## Further reading

- Hyndman, R. J., Athanasopoulos, G., Garza, A., Challu, C., Mergenthaler, M. and Olivares, K. G. (2026). *Forecasting: Principles and Practice, the Pythonic Way*. OTexts. Chapters 2 (time series graphics), 3 (decomposition), 5.2 (simple forecasting methods), 8 (exponential smoothing) and 9 (ARIMA models). https://otexts.com/fpppy/
- statsmodels developers (2026). *Seasonal-Trend decomposition using LOESS (STL)*. https://www.statsmodels.org/stable/examples/notebooks/generated/stl_decomposition.html
- statsmodels developers (2026). *Exponential smoothing* and *ETS models* (example notebooks). https://www.statsmodels.org/stable/examples/notebooks/generated/exponential_smoothing.html
- statsmodels developers (2026). *Time series analysis by state space methods* (`statespace`, including SARIMAX; missing values are handled by the Kalman filter). https://www.statsmodels.org/stable/statespace.html
- Gardner, E. S. (2006). Exponential smoothing: the state of the art, part II. *International Journal of Forecasting*, 22(4), 637–666. https://doi.org/10.1016/j.ijforecast.2006.03.005
- pandas developers (2026). *Time series / date functionality*, section "Resampling". https://pandas.pydata.org/docs/user_guide/timeseries.html#resampling
- Inside Airbnb (2026). *Data assumptions* (how listings, calendars and reviews are scraped). https://insideairbnb.com/data-assumptions/